In [95]:
import numpy as np
import pandas as pd

In [96]:
df = pd.read_csv('cars.csv')

In [97]:
df.sample(5)

,brand,km_driven,fuel,owner,selling_price
7937,Maruti,50000,Diesel,First Owner,760000
99,Hyundai,110000,Diesel,First Owner,725000
2427,Tata,25000,Petrol,First Owner,490000
3938,Jeep,15000,Diesel,First Owner,1515000
3117,Maruti,32995,Petrol,First Owner,445000


In [98]:
df['brand'].value_counts()

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [99]:
df['brand'].nunique()

32

In [100]:
df['fuel'].value_counts()

,count
fuel,
Diesel,4402
Petrol,3631
CNG,57
LPG,38


In [101]:
df['owner'].value_counts()

,count
owner,
First Owner,5289
Second Owner,2105
Third Owner,555
Fourth & Above Owner,174
Test Drive Car,5


## 1. OneHotEncoding

In [102]:
# lets apply one hot encoding over fuel and owner columns
pd.get_dummies(df, columns = ['fuel', 'owner'], dtype = np.int32)

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,0,1,0,0,1,0,0,0,0
1,Skoda,120000,370000,0,1,0,0,0,0,1,0,0
2,Honda,140000,158000,0,0,0,1,0,0,0,0,1
3,Hyundai,127000,225000,0,1,0,0,1,0,0,0,0
4,Maruti,120000,130000,0,0,0,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,0,0,0,1,1,0,0,0,0
8124,Hyundai,119000,135000,0,1,0,0,0,1,0,0,0
8125,Maruti,120000,382000,0,1,0,0,1,0,0,0,0
8126,Tata,25000,290000,0,1,0,0,1,0,0,0,0


In the above, multicollinearity <b>dummy variable trap</b> issue was not solved.

## 2. K-1 OneHotEncoding

Remove 1 column use k-1 encoded columns for each column.

In [103]:
pd.get_dummies(df, columns = ['fuel', 'owner'], drop_first = True)
# fuel_CNG will be removed
# owner_First Owner will be removed

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


##3. One Hot Encoding using sklearn

We cant use pandas.get_dummies() during ml training as pandas doesnot always keeps the order of the encoded features, so its not guaranteed that next time it will remove the same column for (k-1) one hot encoding.

In [104]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 0:4], df.iloc[:,-1],test_size=0.2,random_state=2)

In [105]:
X_train.head()

,brand,km_driven,fuel,owner
5571,Hyundai,35000,Diesel,First Owner
2038,Jeep,60000,Diesel,First Owner
2957,Hyundai,25000,Petrol,First Owner
7618,Mahindra,130000,Diesel,Second Owner
6684,Hyundai,155000,Diesel,First Owner


In [106]:
from sklearn.preprocessing import OneHotEncoder

In [107]:
ohe = OneHotEncoder(drop = 'first', sparse_output = False, dtype = np.int32) # drop first encoded features
# sparse_output = False returns numpy array otherwise we get sparse matrix

In [108]:
X_train_new = ohe.fit_transform(X_train[['fuel', 'owner']])

In [109]:
X_test_new = ohe.transform(X_test[['fuel', 'owner']])

In [110]:
X_train_new

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 0, 0],
       [1, 0, 0, ..., 0, 0, 0]], dtype=int32)

In [111]:
X_train_new.shape

(6502, 7)

In [112]:
# Stack with the remaining features in X_train
X_train_encoded = np.hstack((X_train[['brand', 'km_driven']].values, X_train_new))

In [113]:
pd.DataFrame(X_train_encoded)

,0,1,2,3,4,5,6,7,8
0,Hyundai,35000,1,0,0,0,0,0,0
1,Jeep,60000,1,0,0,0,0,0,0
2,Hyundai,25000,0,0,1,0,0,0,0
3,Mahindra,130000,1,0,0,0,1,0,0
4,Hyundai,155000,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
6497,Ford,35000,1,0,0,0,0,0,0
6498,Maruti,120000,0,0,1,0,0,0,0
6499,Tata,15000,0,0,1,0,0,0,0
6500,Maruti,32500,1,0,0,0,1,0,0


## 4. One Hot Encoding with Top Categories

In [114]:
df['brand'].value_counts()

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [115]:
# There are so many categories in 'brand' feature.
# we want to encode the top categories with higher value counts.
# Least value count categories will be clubed in seperate category 'others'.

In [116]:
counts = df['brand'].value_counts()

In [117]:
df['brand'].nunique()

32

In [118]:
threshold = 100

In [119]:
repl = counts[counts <= threshold].index

In [120]:
repl

Index(['Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi',
       'Lexus', 'Jeep', 'Mitsubishi', 'Land', 'Force', 'Isuzu', 'Ambassador',
       'Kia', 'MG', 'Daewoo', 'Ashok', 'Opel', 'Peugeot'],
      dtype='object', name='brand')

In [121]:
# replace all categories in repl in 'brand' feature with 'uncommon'
pd.get_dummies(df['brand'].replace(repl, 'uncommon'), dtype = np.int32).sample(5)


,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
6875,0,0,0,0,0,1,0,0,0,0,0,0,0
4793,0,0,0,0,0,0,0,0,0,0,1,0,0
614,0,0,0,0,0,0,0,0,0,1,0,0,0
4099,0,0,0,0,0,0,0,0,1,0,0,0,0
6159,0,0,0,0,0,1,0,0,0,0,0,0,0
